In [25]:
import urllib
import os
import zipfile
import pandas as pd
import tensorflow as tf
import numpy as np

In [4]:
def download_and_extract_data():
    url = 'https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset/main/household_power.zip'
    urllib.request.urlretrieve(url, 'household_power.zip')
    with zipfile.ZipFile('household_power.zip', 'r') as zip_ref:
        zip_ref.extractall()

def normalize_series(data, min, max):
    data = data - min
    data = data / max
    return data

def windowed_dataset(series, batch_size, n_past=24, n_future=24, shift=1):
    # YOUR CODE HERE
    dataset = tf.data.Dataset.from_tensor_slices(series)
    dataset = dataset.window(n_past + n_future, shift=shift, drop_remainder=True)
    dataset = dataset.flat_map(lambda window: window.batch(n_past + n_future))
    dataset = dataset.map(lambda window: (window[:n_past], window[n_future:, :1]))
    dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.batch(batch_size).prefetch(1)
    return dataset

In [5]:
download_and_extract_data()
# Reads the dataset from the csv.
df = pd.read_csv('household_power_consumption.csv', sep=',',
                infer_datetime_format=True, index_col='datetime', header=0)
df

C:\Users\ideap\AppData\Local\Temp\ipykernel_13736\2888218216.py:3: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv('household_power_consumption.csv', sep=',',


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...
2007-02-14 17:19:00,0.636,0.140,241.16,2.6,0.0,0.0,0.0
2007-02-14 17:20:00,0.552,0.000,240.46,2.2,0.0,0.0,0.0
2007-02-14 17:21:00,0.538,0.000,239.74,2.2,0.0,0.0,0.0


In [18]:
new_df = df.iloc[0:24]
new_df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
2006-12-16 17:29:00,3.520,0.522,235.02,15.0,0.0,2.0,17.0
2006-12-16 17:30:00,3.702,0.520,235.09,15.8,0.0,1.0,17.0
2006-12-16 17:31:00,3.700,0.520,235.22,15.8,0.0,1.0,17.0
2006-12-16 17:32:00,3.668,0.510,233.99,15.8,0.0,1.0,17.0


In [20]:
new_data = new_df.values
new_data = normalize_series(new_data, new_data.min(axis=0), new_data.max(axis=0))

C:\Users\ideap\AppData\Local\Temp\ipykernel_13736\756891246.py:9: RuntimeWarning: invalid value encountered in divide
  data = data / max


In [21]:
new_data

array([[1.23280561e-01, 7.91666667e-01, 1.62773045e-02, 1.38554217e-01,
                   nan, 5.00000000e-01, 5.55555556e-02],
       [2.71736309e-01, 8.25757576e-01, 1.11748334e-02, 2.77108434e-01,
                   nan, 5.00000000e-01, 0.00000000e+00],
       [2.73553076e-01, 9.43181818e-01, 9.74108122e-03, 2.77108434e-01,
                   nan, 1.00000000e+00, 5.55555556e-02],
       [2.75369842e-01, 9.50757576e-01, 1.16386944e-02, 2.77108434e-01,
                   nan, 5.00000000e-01, 5.55555556e-02],
       [5.19076045e-02, 1.00000000e+00, 1.98195159e-02, 6.02409639e-02,
                   nan, 5.00000000e-01, 5.55555556e-02],
       [3.29613288e-02, 9.88636364e-01, 1.70363498e-02, 3.61445783e-02,
                   nan, 1.00000000e+00, 5.55555556e-02],
       [5.65792889e-02, 9.84848485e-01, 1.73315341e-02, 6.02409639e-02,
                   nan, 5.00000000e-01, 5.55555556e-02],
       [5.63197508e-02, 9.84848485e-01, 1.78797335e-02, 6.02409639e-02,
                   nan, 5

In [6]:
N_FEATURES = 7
BATCH_SIZE = 32  
N_PAST = 24 # Number of past time steps based on which future observations should be predicted
N_FUTURE = 24  # Number of future time steps which are to be predicted.
SHIFT = 1  # By how many positions the window slides to create a new window of observations.

In [22]:
# Normalizes the data
# DO NOT CHANGE THIS
data = df.values
split_time = int(len(data) * 0.5)
data = normalize_series(data, data.min(axis=0), data.max(axis=0))

# Splits the data into training and validation sets.
x_train = data[:split_time]
x_valid = data[split_time:]

In [23]:
new_data = data[0:24]
new_data

array([[0.43377912, 0.47826087, 0.04036551, 0.43564356, 0.        ,
        0.01282051, 0.85      ],
       [0.55716135, 0.49885584, 0.0355582 , 0.54950495, 0.        ,
        0.01282051, 0.8       ],
       [0.55867127, 0.56979405, 0.03420739, 0.54950495, 0.        ,
        0.02564103, 0.85      ],
       [0.56018119, 0.57437071, 0.03599523, 0.54950495, 0.        ,
        0.01282051, 0.85      ],
       [0.37446074, 0.60411899, 0.04370282, 0.37128713, 0.        ,
        0.01282051, 0.85      ],
       [0.35871441, 0.597254  , 0.04108065, 0.35148515, 0.        ,
        0.02564103, 0.85      ],
       [0.3783434 , 0.59496568, 0.04135876, 0.37128713, 0.        ,
        0.01282051, 0.85      ],
       [0.3781277 , 0.59496568, 0.04187525, 0.37128713, 0.        ,
        0.01282051, 0.85      ],
       [0.37467645, 0.58352403, 0.03698848, 0.37128713, 0.        ,
        0.01282051, 0.85      ],
       [0.37402934, 0.58352403, 0.03647199, 0.37128713, 0.        ,
        0.02564103, 0.8

In [24]:
new_data.shape

(24, 7)

In [28]:
new_data = np.array(new_data)
test = new_data.reshape(1, 24, 7)
test.shape

(1, 24, 7)

In [8]:
train_set = windowed_dataset(x_train, BATCH_SIZE, N_PAST, N_FUTURE, SHIFT)
valid_set = windowed_dataset(x_valid, BATCH_SIZE, N_PAST, N_FUTURE, SHIFT)

In [9]:
for X_batch, y_batch in train_set.take(1):
    print(f"Input batch shape (X): {X_batch.shape}")
    print(f"Target batch shape (y): {y_batch.shape}")

Input batch shape (X): (32, 24, 7)
Target batch shape (y): (32, 24, 1)


In [10]:
# Code to define your model.
model = tf.keras.models.Sequential([
    # Whatever your first layer is, the input shape will be (N_PAST = 24, N_FEATURES = 7)
    tf.keras.layers.Dense(150, input_shape=(N_PAST, N_FEATURES), activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(N_FUTURE),
])

In [13]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 24, 150)           1200      
                                                                 
 dense_1 (Dense)             (None, 24, 100)           15100     
                                                                 
 dense_2 (Dense)             (None, 24, 24)            2424      
                                                                 
Total params: 18,724
Trainable params: 18,724
Non-trainable params: 0
_________________________________________________________________


In [14]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if (logs.get('mae') < 0.98):
            print("\nMAE < 0.098, stop training.")
            self.model.stop_training = True

In [16]:
custom_callback = myCallback()

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)

model.compile(loss=tf.keras.losses.Huber(), optimizer=optimizer, metrics=['mae'])

model.fit(train_set, epochs=100, validation_data=valid_set, callbacks=custom_callback)

Epoch 1/100


   1343/Unknown - 4s 3ms/step - loss: 0.0226 - mae: 0.1528
MAE < 0.098, stop training.
1349/1349 [==============================] - 7s 5ms/step - loss: 0.0226 - mae: 0.1527 - val_loss: 0.0183 - val_mae: 0.1331


In [29]:
predictions = model.predict(test)

1/1 [==============================] - 0s 91ms/step


In [30]:
print(predictions)

[[[ 4.07253653e-02  3.70558351e-02  4.22619656e-02  9.11220461e-02
    1.50761619e-01 -1.15869008e-01 -3.33699286e-02  6.35680184e-02
    7.08220378e-02 -1.16165608e-01 -2.36761943e-03  6.49421848e-03
    1.57196671e-02  1.15435570e-03  3.02761644e-02  4.99654338e-02
   -1.54396087e-01  1.77771021e-02  1.33567303e-03 -1.32038798e-02
    1.03247918e-01  5.70562147e-02  2.42415205e-01  6.18719235e-02]
  [ 5.84824681e-02  3.73493470e-02  5.50680980e-02  9.60335135e-02
    1.29508629e-01 -1.12749137e-01 -3.58284488e-02  6.36471361e-02
    5.72183430e-02 -1.33266285e-01 -2.46947184e-02  4.63211164e-03
    3.16593200e-02  2.71360353e-02  1.24320509e-02  6.60662726e-02
   -1.68408558e-01  5.60692325e-03 -4.76073846e-03 -1.92708056e-02
    9.34707224e-02  6.56936765e-02  2.46900290e-01  5.44941500e-02]
  [ 5.54883108e-02  4.05752026e-02  4.95963208e-02  1.01500213e-01
    1.36752814e-01 -1.29352719e-01 -3.85029539e-02  6.74869791e-02
    6.67477399e-02 -1.38079450e-01 -2.54615173e-02 -5.662193

In [ ]:
predictions.shape

(1, 24, 24)

: 